# 5G Network Intelligence — Reproducible Multi-Corpus Encoder Transfer

## Research question
Equal-compute target-self and nine-domain external pretraining, then full-cell fine-tuning across two folds. Does the candidate materially reduce original DLPRB MAE/WAPE against the fixed strong control? Improvement remains an experimental question.


## Temporal and reproducibility contract
Fit/stop/score cutoffs: 335/383/443 and 395/443/503. Context 168 hours, four KPIs/masks, horizons 1/6/24h, all 2,880 original beams in 90 cells. Exclude hours 504 onward and the observed holdout. Historical v1 omitted explicit Torch seeding; its point scores are exploratory, not controlled ablations. This protocol seeds Python, NumPy, CPU Torch and all CUDA generators, saves cold weights before any optimization/transfer, and disables the GPU attention fastpath for consistent stopping/scoring. Historical snapshots are not overwritten.


In [ ]:
# Review Run All never trains.
RUN_TRAINING = False


## Configuration
Seed 17, maximum 100 fine-tuning epochs, patience 12. Only stopping MAE selects weights. Record best/last weights, optimizer, RNG, cold initialization hashes and exact compute counts in durable cloud storage.


In [ ]:
from pathlib import Path
import json
import yaml
import pandas as pd
from IPython.display import display
PROJECT = Path.cwd()
path = PROJECT / 'configs/cell_multisource_v2.yaml'
if path.exists():
    protocol = yaml.safe_load(path.read_text(encoding='utf-8'))
    display(protocol)


## Inputs and model
Use global per-KPI fit-prefix medians/means/scales, observed zeros and missing masks. Raw same-cell beam histories end at the origin; no fitted beam statistics or identity embeddings. Shared temporal patch encoder, spatial attention, causal seasonal residual and nonnegative physical DLPRB output. Local control masks off-diagonal attention.


## External prefix pretraining
Nine domains belong to four collections: three NR bands, CESNET24, four Vigo KPIs and AGG23. Each collection has equal aggregate sampling weight. Use 20 epochs × 65,536 windows / batch 256 = 5,120 updates per pretraining run. Target-self controls receive the same compute and only their original fit prefix. Transfer temporal encoder weights only; discard the dimensionless source head. Normalize source windows/labels from past context. Vigo publisher scaling scope is unknown; affine invariance of nondegenerate normalized windows is not a publisher preprocessing audit. Traffic bytes/sessions are not PRB labels. Telecom Italia is still outside this frozen study pending its own audit.


## Transfer checks
Save source cold initialization, completed pretraining checkpoint, target cold initialization and post-transfer initialization. Verify encoder equality and unchanged target head, spatial weights and physical scaling. Restore the target RNG stream and start a fresh optimizer for fine-tuning.


## Hosted training and debugging
Colab/Kaggle only. Epoch logs show valid target counts and loss. Fine-tuning uses original-unit MAE; source pretraining uses dimensionless MAE. Interpret historical `scaled` trainer field names by stage. An epoch cap is not completed patience.


In [ ]:
if RUN_TRAINING:
    import subprocess, sys
    (PROJECT/'reports').mkdir(exist_ok=True)
    with (PROJECT/'reports/cell_multisource_v2_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.train_cell_multisource_v2'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        TRAIN_EXIT = process.wait()
else:
    print('Review mode: no fitting.')


## Verified evaluation
Reconstruct full-population MAE/WAPE, coverage, zero error and CPU/GPU parity before acceptance. Report every horizon and negative results. Default review does not run selection or recompute fits. One seed and reused development cannot prove fresh generalization.


In [ ]:
prefixes = ['cell_multisource_v2_target_self_pretrain', 'cell_multisource_v2_multi_source_pretrain']
for prefix in prefixes:
    verification = PROJECT/'reports'/f'{prefix}_verification.json'
    scores = PROJECT/'reports'/f'{prefix}_development.csv'
    if verification.exists() and scores.exists():
        evidence = json.loads(verification.read_text())
        if evidence.get('all_metrics_reconstructed'):
            results = pd.read_csv(scores)
            results['WAPE (%)'] = 100*results['wape']
            display(results)
            display(evidence)
    else:
        print(prefix, ': independent verification pending')


## Acceptance
Target at least 5% mean MAE improvement without horizon regression. A 2% initial research gate only qualifies additional seeds and paired time-block uncertainty checks. WAPE below 10% is not established. No holdout reuse, final refit or automatic production promotion.
